Este cuaderno, como su nombre indica, es para realizar el entrenamiento y evaluación de modelos de vídeo.

Los modelos que vamos a trabajar en este cuaderno son los siguientes:
- *MCG-NJU/videomae-base* --> inspirado en cómo aprender los modelos de lenguaje (escondiendo palabras para que el modelo las adivine). Este modelo, toma un video, oculta el 90% de los píxeles (en forma de cubosde espacio-tiempo) y se fuerza a sí mismo a reconstruir lo que falta viendo solo el 10% restante.
- *facebook/timesformer-base-finetuned-k400* --> este modelo soluciona un problema crítico en este tipo de tareas: aplicar la atención (heredado de los transformers) sin tener que hacerlo a cada píxel (lo cual consume mucha memoria). Este modelo primero mira la relación espacial (píxeles dentro de un mismo frame) y luego la relación temporal (el mismo píxel a lo largo de los diferentes frames).
- *google/vivit-b-16x2-kinetics400* --> es la evolución directa de un ViT clásico, los cuales los hemos trabajado en el apartado anterior. Este modelo divide el vídeo en "Tubelets" o tubos 3D. Extrae un bloque de píxeles que atraviesa varios frames de golpe desde la primera capa.  

In [98]:
!pip install -q decord transformers evaluate

In [99]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from decord import VideoReader, cpu
import decord
from tqdm import tqdm
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset
import evaluate
from transformers import (
    AutoImageProcessor,
    AutoModelForVideoClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    AutoProcessor
)
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

In [100]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1. Entrenamiento Train/Valid

## 1.1. Selección del modelo y parámetros

In [101]:
# Descomentar el modelo que se quiera entrenar:
#MODELO_ELEGIDO = "videomae"
#MODELO_ELEGIDO = "timesformer"
MODELO_ELEGIDO = "vivit"

In [102]:
# Diccionario de Checkpoints (Hugging Face)
MODEL_ZOO = {
    "videomae": "MCG-NJU/videomae-base",
    "timesformer": "facebook/timesformer-base-finetuned-k400",
    "vivit": "google/vivit-b-16x2-kinetics400"
}

MODEL_CHECKPOINT = MODEL_ZOO[MODELO_ELEGIDO]

In [103]:
# Los modelos de vídeo suelen requerir un tamaño de clip fijo (ej. 16 o 8 frames)
# Para este experimento, extraemos 16 frames por vídeo para videome y timesformers o 8 frames para el vivit

if MODELO_ELEGIDO == "vivit":
  NUM_FRAMES_CLIP = 8
else:
  NUM_FRAMES_CLIP = 16

BATCH_SIZE = 4       # Este valor hay que mantenerlo bajo para evitar Out Of Memory en CUDA

In [104]:
# RUTAS
RUTA_BASE_VIDEOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/"
CSV_TRAIN_MASTER = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1_master.csv"
OUTPUT_DIR = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision_Temporal/{MODELO_ELEGIDO.upper()}_FineTuned"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [105]:
decord.bridge.set_bridge('torch') # Silencia logs y une Decord con PyTorch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1.2. Carga y Partición de Datos

In [106]:
print(f"Lanzando experimento temporal con: {MODELO_ELEGIDO.upper()}")
df_train_master = pd.read_csv(CSV_TRAIN_MASTER)

# Limpieza rápida
if "Unnamed: 0" in df_train_master.columns:
    df_train_master.drop(columns=["Unnamed: 0"], inplace=True)
if "label_task_3_1_merged" in df_train_master.columns:
    df_train_master.rename(columns={"label_task_3_1_merged": "label"}, inplace=True)
df_train_master["label"] = df_train_master["label"].astype(int)

# 90/10 Split
train_df, val_df = train_test_split(
    df_train_master, test_size=0.10, stratify=df_train_master["label"], random_state=42
)

Lanzando experimento temporal con: VIVIT


In [107]:
def fix_video_paths(df, base_path):
    columna = 'path_video' if 'path_video' in df.columns else 'id_EXIST'
    rutas = []
    for val in df[columna]:
        val_str = str(val)
        if not val_str.endswith(".mp4"): val_str += ".mp4"
        if "videos/" in val_str:
            ruta_completa = os.path.join(base_path, val_str)
        else:
            ruta_completa = os.path.join(base_path, "videos", val_str)
        rutas.append(ruta_completa)
    df['ruta_absoluta'] = rutas
    return df

train_df = fix_video_paths(train_df.copy(), RUTA_BASE_VIDEOS)
val_df = fix_video_paths(val_df.copy(), RUTA_BASE_VIDEOS)

## 1.3. Extracción de Frames y PyTorch dataset

In [108]:
print(f"Cargando procesador para {MODEL_CHECKPOINT}...")

if MODELO_ELEGIDO == "vivit":
  processor = AutoProcessor.from_pretrained(MODEL_CHECKPOINT)
else:
  processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)

Cargando procesador para google/vivit-b-16x2-kinetics400...


In [109]:
def sample_frame_indices(clip_len, total_frames):
    if total_frames <= clip_len:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
    else:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()

In [110]:
class VideoClassificationDataset(TorchDataset):
    def __init__(self, df, processor, clip_len=16):
        self.rutas = df['ruta_absoluta'].tolist()
        self.etiquetas = df['label'].tolist()
        self.processor = processor
        self.clip_len = clip_len

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, idx):
        ruta_video = self.rutas[idx]
        etiqueta = self.etiquetas[idx]

        try:
            vr = VideoReader(ruta_video, ctx=cpu(0))
            frame_indices = sample_frame_indices(self.clip_len, len(vr))
            frames = vr.get_batch(frame_indices).numpy()

            inputs = self.processor(list(frames), return_tensors="pt")
            pixel_values = inputs["pixel_values"][0]

        except Exception as e:
            # Salvavidas: Tensor negro si el vídeo está roto
            pixel_values = torch.zeros((self.clip_len, 3, 224, 224))
            etiqueta = 0

        if MODELO_ELEGIDO == "vivit":
          if pixel_values.shape[0] == 3:
            pixel_values = pixel_values.permute(1, 0, 2, 3)

        return {"pixel_values": pixel_values, "label": etiqueta}

train_dataset = VideoClassificationDataset(train_df, processor, clip_len=NUM_FRAMES_CLIP)
valid_dataset = VideoClassificationDataset(val_df, processor, clip_len=NUM_FRAMES_CLIP)

In [111]:
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['label'] for x in batch])
    }

## 1.4. Modelo y Métricas

In [112]:
id2label = {0: "No Misógino", 1: "Misógino"}
label2id = {"No Misógino": 0, "Misógino": 1}

In [113]:
model = AutoModelForVideoClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    num_frames = NUM_FRAMES_CLIP,
    ignore_mismatched_sizes=True
)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `400`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] VivitForVideoClassification LOAD REPORT from: google/vivit-b-16x2-kinetics400
Key                                  | Status   |                                                                                                  
-------------------------------------+----------+--------------------------------------------------------------------------------------------------
vivit.embeddings.position_embeddings | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 3137, 768]) vs model:torch.Size([1, 785, 768])
classifier.bias                      | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400]) vs model:torch.Size([2])                   
classifier.weight                    | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400, 768]) vs model:torch.Size([2, 768])         

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [114]:
f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

In [115]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}

## 1.5. Entrenamiento

In [116]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False, # CRUCIAL
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    logging_steps=50,
    fp16=True,
    report_to="none",
    dataloader_num_workers=2,
    dataloader_pin_memory=True
)

In [117]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [118]:
print(f"\n🚀 Lanzando el entrenamiento temporal ({MODELO_ELEGIDO})...")
trainer.train()

print("\n💾 Guardando el modelo definitivo...")
trainer.save_model(os.path.join(OUTPUT_DIR, "modelo_final"))
processor.save_pretrained(os.path.join(OUTPUT_DIR, "modelo_final"))
print("✅ ¡Entrenamiento completado!")


🚀 Lanzando el entrenamiento temporal (vivit)...


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.702526,0.684816,0.415286,0.527363
2,0.685800,0.709870,0.477513,0.537313
3,0.638139,0.747304,0.498932,0.512438
4,0.356599,0.886760,0.561916,0.567164
5,0.127523,1.244445,0.546298,0.552239


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['vivit.layers.0.attention.q_proj.weight', 'vivit.layers.0.attention.q_proj.bias', 'vivit.layers.0.attention.k_proj.weight', 'vivit.layers.0.attention.k_proj.bias', 'vivit.layers.0.attention.v_proj.weight', 'vivit.layers.0.attention.v_proj.bias', 'vivit.layers.0.attention.o_proj.weight', 'vivit.layers.0.attention.o_proj.bias', 'vivit.layers.0.layernorm_before.weight', 'vivit.layers.0.layernorm_before.bias', 'vivit.layers.0.layernorm_after.weight', 'vivit.layers.0.layernorm_after.bias', 'vivit.layers.0.mlp.fc1.weight', 'vivit.layers.0.mlp.fc1.bias', 'vivit.layers.0.mlp.fc2.weight', 'vivit.layers.0.mlp.fc2.bias', 'vivit.layers.1.attention.q_proj.weight', 'vivit.layers.1.attention.q_proj.bias', 'vivit.layers.1.attention.k_proj.weight', 'vivit.layers.1.attention.k_proj.bias', 'vivit.layers.1.attention.v_proj.weight', 'vivit.layers.1.attention.v_proj.bias', 'vivit.layers.1.attention.o_proj.weight', 'vivit.layers.1.attent


💾 Guardando el modelo definitivo...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ ¡Entrenamiento completado!


# 2. Resultados contra fichero de test estático

## 2.1. Configuración y rutas


In [119]:
CSV_TEST = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"
CSV_SALIDA = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision_Temporal/{MODELO_ELEGIDO.upper()}_FineTuned/predicciones_{MODELO_ELEGIDO}_test.csv"

## 2.2. Carga de los datos (test) y del modelo

In [120]:
print("Cargando el dataset estático de test...")
test_df = pd.read_csv(CSV_TEST)

# Limpieza de columnas igual que en Train
if "Unnamed: 0" in test_df.columns:
    test_df.drop(columns=["Unnamed: 0"], inplace=True)
if "label_task_3_1_merged" in test_df.columns:
    test_df.rename(columns={"label_task_3_1_merged": "label"}, inplace=True)
test_df["label"] = test_df["label"].astype(int)

Cargando el dataset estático de test...


In [121]:
# Arreglar rutas absolutas (usamos la misma lógica que tenías)
def fix_video_paths(df, base_path):
    columna = 'path_video' if 'path_video' in df.columns else 'id_EXIST'
    rutas = []
    for val in df[columna]:
        if not str(val).endswith(".mp4"): val = str(val) + ".mp4"
        rutas.append(os.path.join(base_path, val))
    df['ruta_absoluta'] = rutas
    return df

test_df = fix_video_paths(test_df, RUTA_BASE_VIDEOS)

In [122]:
# Función de muestreo de frames (se usa en el entrenamiento)
def sample_frame_indices(clip_len, total_frames):
    if total_frames <= clip_len:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
    else:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()

In [124]:
print(f"Cargando procesador y modelo {MODELO_ELEGIDO.upper()} entrenado...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if MODELO_ELEGIDO == "vivit":
  processor = AutoProcessor.from_pretrained(OUTPUT_DIR + "/modelo_final")
else:
  processor = AutoImageProcessor.from_pretrained(OUTPUT_DIR + "/modelo_final")

model = AutoModelForVideoClassification.from_pretrained(OUTPUT_DIR + "/modelo_final").to(device)
model.eval()

print(f"Iniciando inferencia sobre {len(test_df)} vídeos de test...")

Cargando procesador y modelo VIVIT entrenado...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Iniciando inferencia sobre 502 vídeos de test...


# 2.3. Inferencia directa video a video

In [125]:
y_true = []
y_pred = []
resultados_para_csv = []

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    id_vid = row['id_EXIST']
    ruta_video = row['ruta_absoluta']
    true_label = row['label']

    try:
        # Lectura eficiente con Decord
        vr = VideoReader(ruta_video, ctx=cpu(0))
        total_frames = len(vr)
        frame_indices = sample_frame_indices(16, total_frames)

        # ⚠️ CORRECCIÓN AQUÍ: Usamos .numpy() en lugar de .asnumpy()
        frames = vr.get_batch(frame_indices).numpy()

        # Procesamos los 16 fotogramas de golpe
        inputs = processor(list(frames), return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)

        # Inferencia
        with torch.no_grad():
            outputs = model(pixel_values=pixel_values)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
            prob_misogino = probs[1].item()

    except Exception as e:
        print(f"\n⚠️ Error procesando {ruta_video}: {e}")
        # En caso de vídeo corrupto, asumimos incertidumbre (0.5) para no romper el test
        prob_misogino = 0.5

    # 1 si prob > 0.5, sino 0
    prediccion_binaria = 1 if prob_misogino > 0.5 else 0

    y_true.append(true_label)
    y_pred.append(prediccion_binaria)

    # Guardamos para el Ensemble Multimodal
    resultados_para_csv.append({
        "id_EXIST": id_vid,
        "prob_misogino_video": prob_misogino,
        "prediccion_binaria_video": prediccion_binaria,
        "label_real": true_label
    })

  0%|          | 1/502 [00:02<19:47,  2.37s/it]


⚠️ Error procesando /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6832462493189770502.mp4: The size of tensor a (1569) must match the size of tensor b (785) at non-singleton dimension 1


  0%|          | 2/502 [00:04<17:48,  2.14s/it]


⚠️ Error procesando /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6983042234413288710.mp4: The size of tensor a (1569) must match the size of tensor b (785) at non-singleton dimension 1


  1%|          | 3/502 [00:05<15:10,  1.82s/it]

...


⚠️ Error procesando /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6911824130102824198.mp4: The size of tensor a (1569) must match the size of tensor b (785) at non-singleton dimension 1


  7%|▋         | 34/502 [00:45<09:47,  1.26s/it]


⚠️ Error procesando /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6990876582923537669.mp4: The size of tensor a (1569) must match the size of tensor b (785) at non-singleton dimension 1


  7%|▋         | 34/502 [00:46<10:39,  1.37s/it]


KeyboardInterrupt: 

## 2.4. Guardado del CSV de predicción y muestreo de métricas

In [ ]:
# Guardamos el CSV
os.makedirs(os.path.dirname(CSV_SALIDA), exist_ok=True)
pd.DataFrame(resultados_para_csv).to_csv(CSV_SALIDA, index=False)
print(f"\n✅ ¡CSV de {MODELO_ELEGIDO} guardado para el Ensemble en: {CSV_SALIDA}!")

In [ ]:
print("\n" + "="*50)
print(f"🏆 RESULTADOS TEST ESTÁTICO: {MODELO_ELEGIDO} (Análisis Temporal)")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print("\nMatriz de Confusión:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["No Misógino", "Misógino"]))